# 03 · Feature engineering and leakage prevention

Use an explicit schema with automatically separated numeric/categorical predictors. Ordinal ratings remain numeric for a transparent baseline; this assumes equal spacing and is a documented limitation. No target-derived feature or oversampling is used.

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd() if (Path.cwd() / 'src').exists() else Path.cwd().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
import json
import pandas as pd
import numpy as np
from IPython.display import display, Markdown, Image
from attrition.config import FIGURES_DIR, METRICS_DIR, FEATURE_COLUMNS
from attrition.data.load_data import load_data, split_data
from attrition.data.explore import exploration_tables
data = load_data()
X_train, X_test, y_train, y_test = split_data(data)


## Build, then fit on training rows only

Numeric medians, scaling statistics, category modes, and one-hot vocabulary are learned by the pipeline. Each CV clone will learn them independently. Unseen inference categories produce all-zero indicators for that field.

In [2]:
from sklearn.linear_model import LogisticRegression
from attrition.features.preprocess import build_pipeline, feature_exclusions
pipeline = build_pipeline(X_train, LogisticRegression(max_iter=2000, random_state=42))
pipeline.fit(X_train, y_train)
processor = pipeline.named_steps['preprocess']
display(pd.Series(feature_exclusions(X_train), name='reason'))
display(pd.Series(processor.get_feature_names_out(), name='transformed_feature'))

EmployeeNumber    Unique employee identifier; no portable predic...
EmployeeCount     Confirmed constant in training data (1); no va...
Over18            Confirmed constant in training data ('Y'); no ...
StandardHours     Confirmed constant in training data (80); no v...
Name: reason, dtype: str

0                                   Age
1                             DailyRate
2                      DistanceFromHome
3                             Education
4               EnvironmentSatisfaction
5                            HourlyRate
6                        JobInvolvement
7                              JobLevel
8                       JobSatisfaction
9                         MonthlyIncome
10                          MonthlyRate
11                   NumCompaniesWorked
12                    PercentSalaryHike
13                    PerformanceRating
14             RelationshipSatisfaction
15                     StockOptionLevel
16                    TotalWorkingYears
17                TrainingTimesLastYear
18                      WorkLifeBalance
19                       YearsAtCompany
20                   YearsInCurrentRole
21              YearsSinceLastPromotion
22                 YearsWithCurrManager
23            BusinessTravel_Non-Travel
24     BusinessTravel_Travel_Frequently


## Inspect fitted statistics and missing/unseen inputs

Inference cannot update the fitted imputer or encoder. Required fields must be present, but explicit nulls are imputed. The next example modifies a training example solely to verify that the transformation remains usable.

In [3]:
names = processor.transformers_[0][2]
medians = processor.named_transformers_['numeric'].named_steps['imputer'].statistics_
np.testing.assert_allclose(medians, X_train[names].median())
display(pd.DataFrame({'feature': names, 'training_median': medians}))
example = X_train.head(1).copy()
example['JobRole'] = 'Unseen demonstration role'
example['MonthlyIncome'] = np.nan
display(pd.DataFrame(pipeline.predict_proba(example), columns=['No', 'Yes']))

,feature,training_median
0,Age,36.0
1,DailyRate,799.5
2,DistanceFromHome,7.0
3,Education,3.0
4,EnvironmentSatisfaction,3.0
5,HourlyRate,66.0
6,JobInvolvement,3.0
7,JobLevel,2.0
8,JobSatisfaction,3.0
9,MonthlyIncome,5004.5


,No,Yes
0,0.994151,0.005849


## Visual check of the engineered representation

Scaling makes numeric coefficients comparable per training standard deviation, while indicator coefficients remain conditional contrasts. Full one-hot encoding with regularization avoids claiming a unique causal interpretation for each coefficient.

In [4]:
import matplotlib.pyplot as plt
scaled = processor.named_transformers_['numeric'].transform(X_train[names])
fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(scaled[:, names.index('MonthlyIncome')], bins=25, color='#357b89')
ax.set(title='Standardized training MonthlyIncome', xlabel='Training standard deviations', ylabel='Count')
display(fig)
plt.close(fig)

<Figure size 960x360 with 1 Axes>

## Key Findings

The complete pipeline handles missing values and unseen categories, drops documented nonpredictive fields, and preserves human-readable feature names. Its learned medians match training rows exactly. Scaling does not remove income skew or make ordinal scales continuous.

## Next Steps

Compare candidate models with fold-fitted preprocessing and inspect a sensitive-attribute ablation alongside the correlated-feature experiment.